## Librerías necesarias

Antes de empezar, cargamos las librerías que vamos a usar en todo el notebook:

- **pandas**: manejo de tablas de datos (el objeto `stroke` es un `DataFrame`).
- **numpy**: cálculos numéricos (percentiles, estadísticas, redondeos).
- **matplotlib**: motor base para graficar.
- **seaborn**: gráficos estadísticos más prolijos, construidos sobre matplotlib.

Estas librerías ya vienen instaladas en Google Colab, así que no hace falta `pip install`.

In [ ]:
import pandas as pd      # manejo de tablas (DataFrames)
import numpy as np       # cálculos numéricos y estadísticos
import matplotlib.pyplot as plt  # motor base de gráficos
import seaborn as sns    # gráficos estadísticos prolijos

# Estilo visual consistente para todo el notebook
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

# 🧠 Ciencia de Datos II — EDA Progresivo
### Dataset: Stroke Prediction

Vamos a explorar el dataset de forma ordenada, respondiendo **una pregunta concreta por bloque**.
La regla de oro: **no modificar nada antes de entender**.

---
## BLOQUE 1 — Carga y dimensiones del dataset

**¿Qué hacemos acá?**
Cargamos el archivo CSV y vemos qué tamaño tiene la base.

**¿Por qué importa?**
Antes de cualquier análisis hay que saber con cuántos registros y cuántas variables trabajamos.
Un dataset de 100 filas se analiza distinto a uno de 25.000.

In [ ]:
import pandas as pd

# ============================================================
# BLOQUE 1 — CARGA Y DIMENSIONES DEL DATASET
# ============================================================

stroke = pd.read_csv('https://raw.githubusercontent.com/MarisaBerger/ciencia-datos-II/main/data/dataset_04_stroke.csv')

filas, columnas = stroke.shape
print(f"Filas    : {filas}")
print(f"Columnas : {columnas}")

**¿Qué aprendemos?**
El dataset tiene **25.000 pacientes** y **18 variables**.
Esto nos da una primera idea del tamaño de la muestra antes de hacer cualquier análisis.

---
## BLOQUE 2 — Variables del dataset

**¿Qué hacemos acá?**
Listamos los nombres de todas las columnas.

**¿Por qué importa?**
Los nombres ya nos cuentan cosas: podemos detectar variables que fueron codificadas,
variables dummy (one-hot encoding) y la variable objetivo (`target`).

In [ ]:
# ============================================================
# BLOQUE 2 — VARIABLES DEL DATASET
# ============================================================

print("Columnas del dataset:")
for i, col in enumerate(stroke.columns, 1):
    print(f"  {i:2d}. {col}")

**¿Qué aprendemos?**
Se observan dos grupos de variables claramente codificadas:

- `work_type_Govt_job`, `work_type_Private`, `work_type_Self-employed`... → probablemente provienen de una variable original `work_type` convertida a **dummies**
- `smoking_status_Unknown`, `smoking_status_formerly smoked`... → ídem con `smoking_status`

También aparecen variables como `gender_encoded`, `ever_married_encoded`, `Residence_type_encoded` que ya fueron transformadas numéricamente.

Esto nos indica que el dataset fue **preprocesado antes de entregarnos**. Hay que estudiarlo con cuidado.

---
## BLOQUE 3 — Primeras observaciones

**¿Qué hacemos acá?**
Miramos las primeras filas de la tabla.

**¿Por qué importa?**
No es solo "ver la tabla". Sirve para detectar rápidamente:
- cómo están expresados los datos
- presencia de `NaN`
- cantidad de decimales
- variables con valores que parecen raros a primera vista

In [ ]:
# ============================================================
# BLOQUE 3 — PRIMERAS OBSERVACIONES
# ============================================================

stroke.head(10)

**¿Qué aprendemos?**
Ya en las primeras filas se nota algo importante: variables como `hypertension` o `heart_disease`,
que por su nombre uno esperaría que fueran **binarias (0 o 1)**, muestran valores **decimales, negativos y mayores a 1**.

Esto no significa automáticamente que estén mal. Lo anotamos como:

> ⚠️ **Hallazgo a investigar:** variables aparentemente binarias tienen valores continuos.

---
## BLOQUE 4 — Estructura y tipo de dato

**¿Qué hacemos acá?**
Vemos el tipo de dato que Python/pandas asignó a cada columna.

**¿Por qué importa?**
Hay una distinción fundamental que hay que defender:

> **Tipo de dato en pandas ≠ significado estadístico de la variable**

Una columna puede estar guardada como `float64` (numérica) y ser conceptualmente **categórica**.

In [ ]:
# ============================================================
# BLOQUE 4 — ESTRUCTURA Y TIPO DE DATO
# ============================================================

# Información general: tipo de dato, valores no nulos
stroke.info()

In [ ]:
# Ver solo los tipos de dato en formato tabla
pd.DataFrame({
    'Tipo de dato': stroke.dtypes,
    'Valores no nulos': stroke.notna().sum(),
    'Valores nulos': stroke.isna().sum()
})

**¿Qué aprendemos?**
Prácticamente todas las variables son `float64`. Pero eso no significa que todas sean cuantitativas continuas.
Por ejemplo, `target` es numérica pero conceptualmente es **binaria categórica** (0 = sin ACV, 1 = con ACV).

---
## BLOQUE 5 — Clasificación conceptual de las variables

Antes de calcular medias y gráficos, clasificamos cada variable según **qué representa**, no según cómo está guardada.

| Variable | Significado probable | Tipo conceptual |
|---|---|---|
| `age` | Edad | Cuantitativa continua |
| `hypertension` | Hipertensión | Originalmente binaria (transformada) |
| `heart_disease` | Enfermedad cardíaca | Originalmente binaria (transformada) |
| `avg_glucose_level` | Glucemia promedio | Cuantitativa continua |
| `bmi` | Índice de masa corporal | Cuantitativa continua |
| `gender_encoded` | Género | Categórica codificada |
| `ever_married_encoded` | Estado civil | Categórica/binaria codificada |
| `Residence_type_encoded` | Tipo de residencia | Categórica codificada |
| `work_type_*` | Tipo de trabajo | Dummy (one-hot encoding) |
| `smoking_status_*` | Estado tabáquico | Dummy (one-hot encoding) |
| `target` | Presencia de ACV | Binaria (variable objetivo) |

> ⚠️ **Importante:** calcular la media de `gender_encoded` no tiene el mismo sentido fisiológico que calcular la media de `bmi`. Hay que ser cuidadosas con qué interpretamos.

---
## BLOQUE 6 — Valores faltantes

**¿Qué hacemos acá?**
Contamos cuántos `NaN` tiene cada columna, y calculamos el porcentaje.

**¿Por qué importa?**
Un faltante no se analiza solo contando cuántos hay.
Hay que ver si **se concentra en determinados grupos** o si está distribuido al azar.
También hay que buscar "**faltantes disfrazados**": valores como -999, 0 o 999 que en realidad representan "sin dato".

In [ ]:
# ============================================================
# BLOQUE 6 — DATOS FALTANTES
# ============================================================

faltantes = pd.DataFrame({
    'Variable': stroke.columns,
    'Cantidad_NA': stroke.isna().sum().values,
    'Porcentaje_NA': (stroke.isna().mean() * 100).round(2).values
}).sort_values('Cantidad_NA', ascending=False).reset_index(drop=True)

faltantes

**¿Qué aprendemos?**
Varias columnas tienen **exactamente 5.500 faltantes**, lo que representa el **22%** de las observaciones.

Esto no parece casualidad. La pregunta que surge inmediatamente es:

> **¿Son los mismos 5.500 pacientes los que tienen `NaN` en todas esas columnas?**

Si lo fueran, hay un **patrón sistemático de faltantes** que hay que investigar.
Eso lo vamos a ver en el Bloque 9 (análisis detallado de NA).

Por ahora lo anotamos como hallazgo.

---
## BLOQUE 7 — Registros duplicados

**¿Qué hacemos acá?**
Buscamos filas completamente iguales en todas sus columnas.

**¿Por qué importa?**
Un duplicado exacto puede indicar un error al armar el dataset (por ejemplo, un registro cargado dos veces).
Si aparecen muchos, habría que evaluar si eliminarlos o investigar por qué están.

In [ ]:
# ============================================================
# BLOQUE 7 — REGISTROS DUPLICADOS
# ============================================================

duplicados = stroke.duplicated().sum()
print(f"Filas completamente duplicadas: {duplicados}")

**¿Qué aprendemos?**
No hay filas completamente duplicadas.

> ⚠️ **Atención con la expresión "completamente"**: significa que no existe otra fila idéntica en las 18 columnas.
No significa que no pueda haber dos personas con la misma edad, el mismo BMI o la misma glucosa.
Eso sería coincidencia de datos, no un duplicado.

---
## BLOQUE 8 — Distribución de la variable objetivo (`target`)

**¿Qué hacemos acá?**
Contamos cuántos casos hay de cada valor de `target`.

**¿Por qué importa?**
`target` es la variable que queremos predecir (0 = sin ACV, 1 = con ACV).
Si está muy desbalanceada, una simple *accuracy* puede ser engañosa:
un modelo que predice "0" siempre podría tener 87% de accuracy sin aprender nada útil.

In [ ]:
# ============================================================
# BLOQUE 8 — DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# ============================================================

conteo = stroke['target'].value_counts().rename({0: 'Sin ACV (0)', 1: 'Con ACV (1)'})
porcentaje = (stroke['target'].value_counts(normalize=True) * 100).round(2).rename({0: 'Sin ACV (0)', 1: 'Con ACV (1)'})

resumen_target = pd.DataFrame({
    'Casos': conteo,
    'Porcentaje (%)': porcentaje
})

print(resumen_target)
print(f"\nRelación aproximada: {round(conteo['Sin ACV (0)'] / conteo['Con ACV (1)'], 1)} casos sin ACV por cada caso con ACV")

**¿Qué aprendemos?**

| Clase | Casos | % |
|---|---|---|
| Sin ACV (0) | 21.875 | 87,5% |
| Con ACV (1) | 3.125 | 12,5% |

El dataset está **desbalanceado**: hay aproximadamente 7 pacientes sin ACV por cada paciente con ACV.

Esto es importante para modelos de Machine Learning: una accuracy alta no garantiza un buen modelo.
Métricas como precisión, recall o F1-score van a ser más relevantes.

---

## ✅ Cierre del primer bloque de análisis

Con estos 8 bloques ya podemos afirmar:

> *"El dataset tiene 25.000 registros y 18 variables. Incluye variables demográficas, antecedentes clínicos, medidas fisiológicas y variables categóricas previamente codificadas. No presenta filas completamente duplicadas. Varias columnas presentan un 22% de datos faltantes con un patrón a investigar. La variable objetivo está desbalanceada: 87,5% sin ACV vs 12,5% con ACV."*

**Lo que sigue:**
- Bloque 9: Resumen estadístico de variables clínicas (`age`, `bmi`, `avg_glucose_level`)
- Bloque 10: Detección de valores extremos y evaluación fisiológica
- Bloque 11: Análisis del patrón de faltantes (¿son los mismos 5.500 pacientes?)

---
## BLOQUE 9 — Resumen estadístico de variables clínicas

**¿Qué queremos saber?**
Queremos resumir numéricamente cómo se comportan las tres variables cuantitativas continuas del dataset: `age`, `avg_glucose_level` y `bmi`. Un resumen estadístico da una foto rápida de escala, centro y dispersión antes de graficar nada.

**¿Qué hacemos?**
Para cada variable calculamos: cantidad de observaciones válidas, cantidad de NA, mínimo, máximo, media, mediana, desvío estándar, Q1, Q3 e IQR (rango intercuartílico). Armamos una tabla comparativa.

In [ ]:
# ============================================================
# BLOQUE 9 — RESUMEN ESTADÍSTICO DE VARIABLES CLÍNICAS
# ============================================================

variables_clinicas = ['age', 'avg_glucose_level', 'bmi']

resumen_stats = []
for var in variables_clinicas:
    serie = stroke[var]
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    resumen_stats.append({
        'Variable': var,
        'N_validos': serie.notna().sum(),
        'N_NA': serie.isna().sum(),
        'Minimo': serie.min(),
        'Maximo': serie.max(),
        'Media': serie.mean(),
        'Mediana': serie.median(),
        'Desvio_std': serie.std(),
        'Q1': q1,
        'Q3': q3,
        'IQR': q3 - q1
    })

resumen_stats = pd.DataFrame(resumen_stats).round(2)
resumen_stats

**¿Qué observamos?**

- Comparando **media** y **mediana** de cada variable podemos detectar asimetría: si son muy distintas, la distribución probablemente tiene cola larga hacia un lado.
- La **mediana** es más robusta que la media frente a valores extremos, porque no se calcula sumando todos los valores sino ordenando y tomando el del medio: un valor absurdamente alto o bajo no la desplaza tanto como a la media.
- El **IQR** (Q3 − Q1) da el rango donde se concentra el 50% central de los datos, y lo vamos a usar en el Bloque 12 para detectar outliers.

---
## BLOQUE 10 — Distribución de variables cuantitativas

**¿Qué queremos saber?**
Queremos ver la forma real de la distribución de `age`, `avg_glucose_level` y `bmi`: si son simétricas, si tienen cola, dónde se concentran los valores.

**¿Qué hacemos?**
Para cada variable graficamos, por separado, un histograma (forma general) y un boxplot (mediana, cuartiles, valores extremos). Evitamos superponer variables con escalas distintas en un mismo gráfico.

In [ ]:
# ============================================================
# BLOQUE 10 — DISTRIBUCIÓN DE VARIABLES CUANTITATIVAS
# ============================================================

unidades = {'age': 'años', 'avg_glucose_level': 'mg/dL', 'bmi': 'kg/m²'}

for var in variables_clinicas:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(stroke[var].dropna(), bins=40, color='#4C72B0', edgecolor='white')
    axes[0].set_title(f'Histograma de {var}')
    axes[0].set_xlabel(f'{var} ({unidades[var]})')
    axes[0].set_ylabel('Frecuencia')

    axes[1].boxplot(stroke[var].dropna(), vert=True, labels=[var])
    axes[1].set_title(f'Boxplot de {var}')
    axes[1].set_ylabel(f'{var} ({unidades[var]})')

    plt.tight_layout()
    plt.show()

**¿Qué observamos?**
Los histogramas permiten ver si cada variable es más simétrica (como suele esperarse en `age`) o si tiene cola hacia la derecha (como suele pasar con `avg_glucose_level` y `bmi`, donde una minoría de valores altos estira la distribución). Los boxplots muestran esa misma asimetría a través de la posición de la mediana dentro de la caja y la cantidad de puntos que aparecen por fuera de los bigotes (candidatos a outliers, que analizamos en detalle en el Bloque 12).

---
## BLOQUE 11 — Boxplots comparables

**¿Qué queremos saber?**
Queremos ver si tiene sentido comparar visualmente `age`, `avg_glucose_level` y `bmi` en un mismo gráfico.

**¿Qué hacemos?**
Primero graficamos las tres variables juntas sin ningún ajuste: como están en escalas muy distintas (edad ~0-100 años, glucosa ~50-300 mg/dL, BMI ~10-60 kg/m²), el resultado es engañoso. Después mostramos una versión donde estandarizamos (z-score) **solo para graficar**, dejando aclarado que esto no modifica el dataset `stroke` original.

In [ ]:
# ============================================================
# BLOQUE 11 — BOXPLOTS COMPARABLES
# ============================================================

# Versión SIN ajustar (escalas originales, mezcladas)
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([stroke[var].dropna() for var in variables_clinicas], labels=variables_clinicas)
ax.set_title('Boxplots en escala original (NO comparables)')
ax.set_ylabel('Valor original (unidades distintas por variable)')
plt.show()

# Versión estandarizada SOLO para graficar (no modifica 'stroke')
estandarizado = pd.DataFrame({
    var: (stroke[var] - stroke[var].mean()) / stroke[var].std()
    for var in variables_clinicas
})

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([estandarizado[var].dropna() for var in variables_clinicas], labels=variables_clinicas)
ax.set_title('Boxplots estandarizados (z-score) — solo para visualización')
ax.set_ylabel('Valor estandarizado (media 0, desvío 1)')
plt.show()

**¿Qué observamos?**
En la versión sin ajustar, `avg_glucose_level` domina visualmente el gráfico por tener la escala más grande, lo que hace parecer que `age` y `bmi` casi no varían — una lectura engañosa. La versión estandarizada (z-score) permite comparar la **forma relativa** de las tres distribuciones (dispersión, asimetría, outliers) en una escala común. Importante: `estandarizado` es una tabla auxiliar creada solo para este gráfico; el DataFrame `stroke` no fue modificado.

---
## BLOQUE 12 — Outliers (criterio IQR)

**¿Qué queremos saber?**
Queremos identificar, de forma objetiva y reproducible, cuántos valores de `age`, `avg_glucose_level` y `bmi` caen fuera del rango "esperable" según el criterio estadístico del rango intercuartílico (IQR).

**¿Qué hacemos?**
Para cada variable calculamos Q1, Q3, IQR, y los límites: `Límite inferior = Q1 - 1.5·IQR` y `Límite superior = Q3 + 1.5·IQR`. Contamos cuántos valores quedan fuera de esos límites. **No los eliminamos ni modificamos**, solo los identificamos.

In [ ]:
# ============================================================
# BLOQUE 12 — OUTLIERS SEGÚN CRITERIO IQR
# ============================================================

outliers_iqr = []
outliers_por_variable = {}  # índices de outliers por variable, para reutilizar más adelante

for var in variables_clinicas:
    serie = stroke[var].dropna()
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr

    es_outlier = (serie < lim_inf) | (serie > lim_sup)
    cantidad = int(es_outlier.sum())

    outliers_por_variable[var] = serie[es_outlier].index

    outliers_iqr.append({
        'Variable': var,
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'IQR': round(iqr, 2),
        'Limite_inferior': round(lim_inf, 2),
        'Limite_superior': round(lim_sup, 2),
        'Cantidad_outliers': cantidad,
        'Porcentaje_outliers': round(cantidad / len(serie) * 100, 2)
    })

outliers_iqr = pd.DataFrame(outliers_iqr)
outliers_iqr

**¿Qué observamos?**
La tabla muestra, para cada variable, dónde el criterio IQR ubica el límite "esperable" y cuántas observaciones quedan afuera. Este criterio es puramente estadístico: no sabe nada de fisiología. Por eso en el Bloque 13 vamos a mirar esos valores extremos concretos y preguntarnos si, además de ser estadísticamente atípicos, son fisiológicamente posibles.

---
## BLOQUE 13 — Outliers estadísticos vs errores de registro

**¿Qué queremos saber?**
Un outlier estadístico no es automáticamente un error. Queremos revisar los valores extremos reales del dataset y evaluarlos desde dos ángulos distintos: ¿es estadísticamente atípico? y, por separado, ¿es fisiológicamente posible?

**¿Qué hacemos?**
Para cada variable clínica mostramos los valores más bajos y más altos que aparecen en el dataset, y armamos una tabla de hallazgos combinando el criterio estadístico (Bloque 12) con reglas fisiológicas razonables. Cuando no podemos estar seguras del origen del valor, lo marcamos como "requiere revisión" en lugar de inventar una explicación.

In [ ]:
# ============================================================
# BLOQUE 13 — VALORES EXTREMOS REALES POR VARIABLE
# ============================================================

for var in variables_clinicas:
    print(f"--- {var} ---")
    print("Mínimos:", sorted(stroke[var].dropna().nsmallest(5).tolist()))
    print("Máximos:", sorted(stroke[var].dropna().nlargest(5).tolist()))
    print()

In [ ]:
# Reglas fisiológicas razonables (no arbitrarias) para evaluar cada valor extremo.
# Cuando la evaluación no es concluyente, se marca como "requiere revisión".
def evaluar_fisiologia(var, valor):
    if pd.isna(valor):
        return 'N/A'
    if var == 'age':
        return 'Fisiológicamente imposible' if (valor < 0 or valor > 120) else 'Fisiológicamente posible'
    if var == 'bmi':
        if valor <= 0:
            return 'Fisiológicamente imposible'
        return 'Extremo, requiere revisión' if valor > 80 else 'Fisiológicamente posible'
    if var == 'avg_glucose_level':
        if valor <= 0:
            return 'Fisiológicamente imposible'
        return 'Extremo pero posible / requiere revisión' if valor > 300 else 'Fisiológicamente posible'
    return 'requiere revisión'

hallazgos = []
for var in variables_clinicas:
    serie = stroke[var].dropna()
    lim_inf = outliers_iqr.loc[outliers_iqr['Variable'] == var, 'Limite_inferior'].values[0]
    lim_sup = outliers_iqr.loc[outliers_iqr['Variable'] == var, 'Limite_superior'].values[0]

    for etiqueta, valor in [('Mínimo observado', serie.min()), ('Máximo observado', serie.max())]:
        es_outlier_stat = (valor < lim_inf) or (valor > lim_sup)
        hallazgos.append({
            'Variable': var,
            'Valor_extremo': round(valor, 2),
            'Tipo': etiqueta,
            'Outlier_estadistico': 'Sí' if es_outlier_stat else 'No',
            'Fisiologicamente_posible': evaluar_fisiologia(var, valor),
        })

hallazgos = pd.DataFrame(hallazgos)
hallazgos

**¿Qué observamos?**
La tabla combina dos preguntas independientes para cada valor extremo: si es estadísticamente atípico (criterio IQR del Bloque 12) y si es fisiológicamente posible (según reglas médicas razonables, no arbitrarias). Un valor puede ser estadísticamente atípico y perfectamente posible (por ejemplo, una persona muy longeva), o puede ser fisiológicamente imposible aunque no siempre resulte matemáticamente "extremo" (por ejemplo, un BMI negativo). Cuando la evaluación fisiológica no es concluyente, la marcamos como "requiere revisión" en vez de suponer un origen del error que no podemos comprobar con los datos disponibles.

---
## BLOQUE 14 — Detección de valores imposibles o sospechosos

**¿Qué queremos saber?**
Más allá del criterio estadístico (IQR), queremos aplicar reglas de validación explícitas y biológicamente fundamentadas: ¿hay valores que son directamente imposibles, independientemente de si son "outliers" o no?

**¿Qué hacemos?**
Definimos reglas concretas (`age < 0`, `age > 120`, `bmi <= 0`, `avg_glucose_level <= 0`) y contamos cuántos registros las violan. Esto es distinto de los outliers estadísticos del Bloque 12: un valor puede ser estadísticamente atípico sin ser imposible, y viceversa.

In [ ]:
# ============================================================
# BLOQUE 14 — VALORES IMPOSIBLES O SOSPECHOSOS
# ============================================================

reglas = {
    'age < 0': stroke['age'] < 0,
    'age > 120': stroke['age'] > 120,
    'bmi <= 0': stroke['bmi'] <= 0,
    'avg_glucose_level <= 0': stroke['avg_glucose_level'] <= 0,
}

reporte_reglas = pd.DataFrame({
    'Regla': list(reglas.keys()),
    'Cantidad_registros': [int(cond.sum()) for cond in reglas.values()],
    'Porcentaje': [round(cond.mean() * 100, 3) for cond in reglas.values()],
})

reporte_reglas

**¿Qué observamos?**
Esta tabla separa conceptualmente tres categorías que suelen confundirse:
1. **Valores imposibles**: violan una regla biológica dura (por ejemplo, `bmi <= 0`).
2. **Valores muy extremos pero posibles**: por ejemplo, una `age` cercana a 100 años.
3. **Outliers estadísticos comunes**: ya identificados en el Bloque 12, que no necesariamente violan ninguna regla biológica.

Cuántos registros aparecen en esta tabla (si aparecen) determina qué tan urgente es revisar el origen del dataset antes de avanzar hacia una eventual limpieza o modelado.

---
## BLOQUE 15 — Variables binarias/categóricas codificadas

**¿Qué queremos saber?**
Varias columnas del dataset (`hypertension`, `heart_disease`, `gender_encoded`, `ever_married_encoded`, `Residence_type_encoded`, y las dummies `work_type_*` / `smoking_status_*`) deberían, en principio, tomar solo los valores 0 y 1. Queremos comprobar si realmente es así.

**¿Qué hacemos?**
Para cada una de estas variables mostramos sus valores únicos, frecuencia y porcentaje. Además, el código detecta automáticamente si aparece algún valor fuera de {0, 1} (negativo, decimal, o mayor a 1) y lo marca como **"⚠️ valor a investigar"**, sin corregirlo.

In [ ]:
# ============================================================
# BLOQUE 15 — VARIABLES BINARIAS/CATEGÓRICAS CODIFICADAS
# ============================================================

cols_work_type = stroke.columns[stroke.columns.str.startswith('work_type_')].tolist()
cols_smoking = stroke.columns[stroke.columns.str.startswith('smoking_status_')].tolist()

variables_codificadas = (
    ['hypertension', 'heart_disease', 'gender_encoded', 'ever_married_encoded', 'Residence_type_encoded']
    + cols_work_type + cols_smoking
)

variables_sospechosas = []  # variables con valores fuera de {0, 1}

for var in variables_codificadas:
    conteo = stroke[var].value_counts(dropna=False).sort_index()
    porcentaje = (stroke[var].value_counts(normalize=True, dropna=False).sort_index() * 100).round(2)
    tabla = pd.DataFrame({'Frecuencia': conteo, 'Porcentaje (%)': porcentaje})

    valores_fuera_de_rango = [v for v in conteo.index if pd.notna(v) and v not in (0, 1)]

    print(f"--- {var} ---")
    print(tabla)
    if valores_fuera_de_rango:
        print(f"⚠️ valor a investigar: valores fuera de 0/1 → {valores_fuera_de_rango}")
        variables_sospechosas.append(var)
    print()

**¿Qué observamos?**
Si el aviso "⚠️ valor a investigar" aparece en alguna variable, significa que esa columna —pese a su nombre y pese a que "debería" ser binaria— contiene valores negativos, decimales o mayores a 1. Esto es consistente con el hallazgo del Bloque 3: el dataset probablemente fue **transformado o normalizado** antes de entregarse (por ejemplo, con alguna técnica de escalado), y los nombres de columna ya no reflejan fielmente una codificación 0/1 simple. No corregimos estos valores acá: solo los dejamos documentados en la lista `variables_sospechosas`.

---
## BLOQUE 16 — Patrón de datos faltantes (en profundidad)

**¿Qué queremos saber?**
En el Bloque 6 vimos que varias columnas tienen exactamente ~5.500 valores faltantes (22%). Ahora respondemos la pregunta que quedó pendiente: **¿son los mismos pacientes los que faltan en todas esas columnas, o son faltantes independientes entre sí?**

**¿Qué hacemos?**
Identificamos qué columnas tienen ese patrón de ~5.500 NA, comparamos los conjuntos de índices con NA entre columnas (unión vs intersección), y contamos cuántos NA tiene cada fila en total para ver si los faltantes se concentran en un subconjunto de pacientes. No imputamos nada todavía.

In [ ]:
# ============================================================
# BLOQUE 16 — PATRÓN DE FALTANTES
# ============================================================

# Columnas con un volumen de NA similar al hallazgo del Bloque 6 (entre 20% y 25%)
pct_na_col = stroke.isna().mean() * 100
cols_alto_na = pct_na_col[(pct_na_col > 20) & (pct_na_col < 25)].index.tolist()
print(f"Columnas con ~22% de NA: {cols_alto_na}\n")

# Índices (pacientes) con NA en cada una de esas columnas
sets_na = {col: set(stroke.index[stroke[col].isna()]) for col in cols_alto_na}

interseccion = set.intersection(*sets_na.values()) if sets_na else set()
union = set.union(*sets_na.values()) if sets_na else set()

print(f"Pacientes con NA en al menos una de esas columnas: {len(union)}")
print(f"Pacientes con NA en TODAS esas columnas a la vez : {len(interseccion)}")

# Cantidad de NA por fila (considerando todas las columnas del dataset)
na_por_fila = stroke.isna().sum(axis=1)
distribucion_na_por_fila = na_por_fila.value_counts().sort_index()
print("\nDistribución de cantidad de NA por fila:")
print(distribucion_na_por_fila)

**¿Qué observamos?**
Si el tamaño de la intersección es igual (o muy cercano) al tamaño de la unión, significa que **son básicamente los mismos pacientes** los que tienen NA en todas esas columnas: hay un patrón sistemático (por ejemplo, un bloque entero de mediciones que no se completó para ese subgrupo). Si en cambio la intersección es mucho más chica que la unión, los faltantes son más independientes entre columnas. La distribución de NA por fila muestra si el problema se concentra en pocos pacientes con muchos datos faltantes, o si está repartido de forma más pareja. En ningún caso imputamos estos valores todavía: solo los caracterizamos.

---
## BLOQUE 17 — Correlaciones

**¿Qué queremos saber?**
Entre las variables cuantitativas continuas (`age`, `avg_glucose_level`, `bmi`), ¿existe alguna asociación lineal?

**¿Qué hacemos?**
Calculamos la matriz de correlación de Pearson **solo** entre estas tres variables y la visualizamos con un heatmap. No calculamos correlaciones con las variables categóricas codificadas (`gender_encoded`, `work_type_*`, etc.): aunque están guardadas como números, son códigos de categorías, y una correlación de Pearson entre códigos categóricos no tiene una interpretación conceptual válida.

In [ ]:
# ============================================================
# BLOQUE 17 — CORRELACIONES (SOLO VARIABLES CUANTITATIVAS CONTINUAS)
# ============================================================

matriz_corr = stroke[variables_clinicas].corr()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(matriz_corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Correlación entre variables clínicas continuas')
plt.tight_layout()
plt.show()

matriz_corr

**¿Qué observamos?**
Los valores de la matriz van de -1 a 1: cercanos a 0 indican poca o ninguna asociación lineal, y valores cercanos a ±1 indican una asociación lineal fuerte. **Importante: correlación no implica causalidad.** Que dos variables estén correlacionadas no significa que una cause la otra; puede deberse a una tercera variable, a coincidencia, o a estructura propia del dataset.

---
## BLOQUE 18 — Relación con la variable objetivo (`target`)

**¿Qué queremos saber?**
De forma puramente exploratoria (todavía sin construir ningún modelo): ¿cómo se distribuyen `age`, `avg_glucose_level` y `bmi` entre los pacientes con `target = 0` (sin ACV) y `target = 1` (con ACV)? Preguntas concretas:
- ¿Cómo se distribuye la edad entre ambos grupos?
- ¿Hay diferencias visibles en la glucosa promedio?
- ¿Cómo se comporta el BMI en cada grupo?

**¿Qué hacemos?**
Para cada variable clínica armamos un boxplot separado por `target`, y calculamos estadísticas resumen (media, mediana, desvío) agrupadas por `target`. Esto es exploración de patrones, **no** un análisis causal ni un modelo predictivo.

In [ ]:
# ============================================================
# BLOQUE 18 — RELACIÓN CON LA VARIABLE OBJETIVO
# ============================================================

for var in variables_clinicas:
    fig, ax = plt.subplots(figsize=(6, 4))
    datos_por_grupo = [stroke.loc[stroke['target'] == g, var].dropna() for g in [0, 1]]
    ax.boxplot(datos_por_grupo, labels=['target = 0 (sin ACV)', 'target = 1 (con ACV)'])
    ax.set_title(f'{var} según target')
    ax.set_ylabel(f'{var} ({unidades[var]})')
    plt.tight_layout()
    plt.show()

resumen_por_target = stroke.groupby('target')[variables_clinicas].agg(['mean', 'median', 'std']).round(2)
resumen_por_target

**¿Qué observamos?**
Los boxplots y la tabla permiten comparar visual y numéricamente ambos grupos. Si las medianas o las cajas aparecen claramente desplazadas entre `target = 0` y `target = 1`, eso sugiere una asociación exploratoria entre esa variable y el ACV. **Esto no prueba causalidad ni predice nada todavía**: es solo el primer paso descriptivo antes de cualquier modelado formal.

---
## BLOQUE 19 — Visualizaciones principales del EDA

**¿Qué queremos saber?**
Para una presentación oral no queremos repetir todos los gráficos del notebook: queremos un puñado que resuma lo esencial.

**¿Qué hacemos?**
Elegimos 4 visualizaciones que condensan los hallazgos más importantes: (1) distribución de `target`, (2) distribución de una variable clínica principal, (3) mapa de valores faltantes por columna, (4) comparación de una variable clínica según `target`.

In [ ]:
# ============================================================
# BLOQUE 19 — VISUALIZACIONES PRINCIPALES DEL EDA
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. Distribución de target
conteo_target = stroke['target'].value_counts().sort_index()
axes[0, 0].bar(['Sin ACV (0)', 'Con ACV (1)'], conteo_target.values, color=['#4C72B0', '#C44E52'])
axes[0, 0].set_title('1. Distribución de target')
axes[0, 0].set_ylabel('Cantidad de pacientes')

# 2. Distribución de bmi (variable clínica representativa)
axes[0, 1].hist(stroke['bmi'].dropna(), bins=40, color='#55A868', edgecolor='white')
axes[0, 1].set_title('2. Distribución de bmi')
axes[0, 1].set_xlabel('bmi (kg/m²)')
axes[0, 1].set_ylabel('Frecuencia')

# 3. Valores faltantes por columna
faltantes_ordenado = stroke.isna().sum().sort_values(ascending=False)
faltantes_ordenado = faltantes_ordenado[faltantes_ordenado > 0]
axes[1, 0].barh(faltantes_ordenado.index, faltantes_ordenado.values, color='#8172B2')
axes[1, 0].set_title('3. Cantidad de NA por columna')
axes[1, 0].set_xlabel('Cantidad de NA')
axes[1, 0].invert_yaxis()

# 4. avg_glucose_level según target
datos_por_grupo = [stroke.loc[stroke['target'] == g, 'avg_glucose_level'].dropna() for g in [0, 1]]
axes[1, 1].boxplot(datos_por_grupo, labels=['target=0', 'target=1'])
axes[1, 1].set_title('4. avg_glucose_level según target')
axes[1, 1].set_ylabel('avg_glucose_level (mg/dL)')

plt.tight_layout()
plt.show()

**¿Qué observamos?**
Estos cuatro gráficos condensan, en una sola vista, el desbalance de clases, la forma de una variable clínica clave, el patrón de datos faltantes por columna y una comparación exploratoria entre grupos. Son los gráficos recomendados para una presentación oral del EDA.

---
## BLOQUE 20 — Resumen automático de hallazgos

**¿Qué queremos saber?**
Queremos cerrar el EDA con una lista de hallazgos objetivos, calculados directamente a partir de lo que hicimos en los bloques anteriores, no un texto libre inventado.

**¿Qué hacemos?**
Reunimos en un solo bloque de código los números clave ya calculados: dimensiones del dataset, faltantes, duplicados, distribución de target, outliers, valores imposibles y variables codificadas sospechosas.

In [ ]:
# ============================================================
# BLOQUE 20 — RESUMEN AUTOMÁTICO DE HALLAZGOS
# ============================================================

print("=" * 60)
print("RESUMEN DE HALLAZGOS DEL EDA")
print("=" * 60)

print(f"\n1. Observaciones y variables : {stroke.shape[0]} filas, {stroke.shape[1]} columnas")

pct_na_total = stroke.isna().values.mean() * 100
print(f"2. Porcentaje global de NA   : {pct_na_total:.2f}%")

top_na = faltantes.sort_values('Porcentaje_NA', ascending=False).head(3)
print("3. Variables con más NA      :")
for _, fila in top_na.iterrows():
    print(f"   - {fila['Variable']}: {fila['Porcentaje_NA']}%")

print(f"4. Registros duplicados      : {duplicados}")

print("5. Distribución de target    :")
for _, fila in resumen_target.iterrows():
    print(f"   - {fila.name}: {fila['Casos']} casos ({fila['Porcentaje (%)']}%)")

print("6. Outliers (criterio IQR)   :")
for _, fila in outliers_iqr.iterrows():
    print(f"   - {fila['Variable']}: {fila['Cantidad_outliers']} outliers ({fila['Porcentaje_outliers']}%)")

print("7. Valores imposibles detectados:")
for _, fila in reporte_reglas.iterrows():
    estado = f"{fila['Cantidad_registros']} registros" if fila['Cantidad_registros'] > 0 else "ninguno"
    print(f"   - {fila['Regla']}: {estado}")

print(f"8. Variables codificadas con valores sospechosos (fuera de 0/1): "
      f"{variables_sospechosas if variables_sospechosas else 'ninguna'}")

**¿Qué observamos?**
Este resumen no agrega información nueva: solo reorganiza, en un solo lugar, los resultados numéricos ya calculados a lo largo del notebook. Sirve como base objetiva para armar las conclusiones de una presentación oral, sin necesidad de repetir cada bloque.

---

## Cierre del EDA (Bloques 1-20)

Con este notebook completamos un análisis exploratorio exhaustivo del dataset de stroke: carga, estructura, calidad de datos, distribuciones, outliers, valores sospechosos y relación con la variable objetivo.

**Lo que NO hicimos todavía, a propósito:**
- No imputamos valores faltantes.
- No eliminamos outliers.
- No transformamos ni normalizamos ninguna variable original.
- No entrenamos ningún modelo predictivo.

Estos pasos son la continuación natural del proyecto, una vez que el dataset fue completamente comprendido.